# Flood Model Training

This notebook trains Random Forest and XGBoost flood-event classifiers from the combined rainfall and river dataset.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from flood_pipeline import OUTPUT_DIR, build_dataset, train_models

pd.set_option('display.max_columns', 100)
OUTPUT_DIR.mkdir(exist_ok=True)

## Load Modeling Dataset

In [2]:
dataset_path = OUTPUT_DIR / 'combined_flood_dataset.csv'
if dataset_path.exists():
    df = pd.read_csv(dataset_path, parse_dates=['date', 'event_start', 'event_end'])
else:
    df = build_dataset(fetch_soil=False)
    df.to_csv(dataset_path, index=False)

df.shape

C:\Users\anish\AppData\Local\Temp\ipykernel_16272\419683476.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_path, parse_dates=['date', 'event_start', 'event_end'])


(263159, 30)

In [3]:
preview_cols = ['state', 'district', 'date', 'rainfall_mm', 'river_daily_mean', 'is_flood_event']
df[[c for c in preview_cols if c in df.columns]].head()

,state,district,date,rainfall_mm,river_daily_mean,is_flood_event
0,Assam,BAKSA,2019-01-01,0.0,NaN,0
1,Assam,BAKSA,2019-01-02,0.0,NaN,0
2,Assam,BAKSA,2019-01-03,0.0,NaN,0
3,Assam,BAKSA,2019-01-04,0.0,NaN,0
4,Assam,BAKSA,2019-01-05,0.0,NaN,0


## Target Balance and Temporal Split

In [4]:
target_balance = df['is_flood_event'].value_counts(normalize=True).rename('fraction').to_frame()
target_balance['rows'] = df['is_flood_event'].value_counts()
target_balance

,fraction,rows
is_flood_event,,
0,0.875566,230413
1,0.124434,32746


In [5]:
split_preview = (
    df[df['year'].between(2019, 2023)]
    .assign(split=lambda x: x['year'].where(x['year'] == 2023, 'train'))
    .groupby(['split', 'state'], as_index=False)
    .agg(rows=('is_flood_event', 'size'), flood_rows=('is_flood_event', 'sum'))
)
split_preview

,split,state,rows,flood_rows
0,2023,Assam,10000,2127
1,2023,Kerala,5110,0
2,2023,Odisha,10000,1315
3,2023,Uttar Pradesh,26645,3431
4,train,Assam,42045,8184
5,train,Kerala,20454,2338
6,train,Odisha,40950,4849
7,train,Uttar Pradesh,107955,10502


## Train Random Forest and XGBoost

The reusable training function uses a stratified sample by default so the notebook runs on a normal laptop. Increase `max_train_rows` and `max_test_rows` if you want a longer full-data experiment.

In [6]:
train_models(df, max_train_rows=60000, max_test_rows=40000)
metrics_path = OUTPUT_DIR / 'models' / 'model_metrics.json'
metrics = json.loads(metrics_path.read_text())
metrics['split']

{'train_rows': 60000,
 'test_rows': 40000,
 'train_years': [2019, 2020, 2021, 2022],
 'test_years': [2023],
 'features': ['rainfall_mm',
  'rainfall_roll3_mean',
  'rainfall_roll3_sum',
  'rainfall_roll7_mean',
  'rainfall_roll7_sum',
  'rainfall_roll14_mean',
  'rainfall_roll14_sum',
  'rainfall_lag1',
  'rainfall_lag3',
  'rainfall_lag7',
  'river_daily_mean',
  'river_daily_max',
  'river_daily_min',
  'river_obs_count',
  'river_daily_mean_lag1',
  'river_daily_mean_roll3_mean',
  'river_daily_max_lag1',
  'river_daily_max_roll3_mean',
  'river_is_discharge',
  'month',
  'day_of_year',
  'state',
  'district']}

## Model Metrics

In [7]:
summary_rows = []
for model_name in ['random_forest', 'xgboost']:
    report = metrics[model_name]['classification_report']
    summary_rows.append({
        'model': model_name,
        'roc_auc': metrics[model_name]['roc_auc'],
        'accuracy': report['accuracy'],
        'flood_precision': report['1']['precision'],
        'flood_recall': report['1']['recall'],
        'flood_f1': report['1']['f1-score'],
    })
pd.DataFrame(summary_rows)

,model,roc_auc,accuracy,flood_precision,flood_recall,flood_f1
0,random_forest,0.978151,0.918100,0.625555,0.954819,0.755887
1,xgboost,0.966391,0.904425,0.582057,0.994164,0.734237


In [8]:
pd.DataFrame(metrics['random_forest']['confusion_matrix'], index=['actual_0', 'actual_1'], columns=['pred_0', 'pred_1'])

,pred_0,pred_1
actual_0,31652,3036
actual_1,240,5072


In [9]:
pd.DataFrame(metrics['xgboost']['confusion_matrix'], index=['actual_0', 'actual_1'], columns=['pred_0', 'pred_1'])

,pred_0,pred_1
actual_0,30896,3792
actual_1,31,5281


## Feature Importance

In [10]:
rf_importance = pd.read_csv(OUTPUT_DIR / 'models' / 'random_forest_feature_importance.csv')
xgb_importance = pd.read_csv(OUTPUT_DIR / 'models' / 'xgboost_feature_importance.csv')
rf_importance.head(15)

,feature,importance
0,num__day_of_year,0.275546
1,num__month,0.202534
2,num__rainfall_roll14_sum,0.087807
3,num__rainfall_roll7_mean,0.058123
4,num__rainfall_roll7_sum,0.055448
5,num__rainfall_roll14_mean,0.046174
6,num__rainfall_mm,0.034923
7,num__rainfall_lag1,0.032728
8,num__rainfall_lag7,0.032536
9,num__rainfall_roll3_sum,0.030480


In [11]:
xgb_importance.head(15)

,feature,importance
0,num__month,0.263759
1,num__day_of_year,0.192973
2,cat__state_Assam,0.078439
3,cat__state_Odisha,0.071720
4,cat__state_Uttar Pradesh,0.052544
5,num__river_daily_min,0.044129
6,num__rainfall_roll14_mean,0.032027
7,num__rainfall_roll14_sum,0.031259
8,cat__state_Kerala,0.027956
9,num__rainfall_roll7_mean,0.025793
